# Analysis: Bolt, Clevis, Lug 
According to Roloff / Matek 2017  
**Author**: Artjom Lukanowski  
**Date**: 12.06.2026  

### Input: Generic data

In [176]:
import  math


""" INTEGRATION CASES
 c1: Clearance-fit everywhere 
 c2: Press-fit clevis <-> bolt, clearance bar <-> bolt 
 c3: Clearance clevis <-> bolt, press-fit bar <-> bolt """

int_case = "c1"

""" LOAD CASES
 stat: static
 dyn_pul: pulsating
 dyn_al: alternating """

ld_case = "stat"

""" BOLT HEAT TREATMENT 
ht: heat treatment
no_ht: no heat treatment """

bt_ht ="ht"

# MATERIAL PAIRING

mat_pair = "ST"

# FORCE
F_stat= 1.5*15000  #[N]

# FACTORS
#Clamping factor. Depending on integration case
if int_case == "c1": 
    k=1.6
else: k = 1.1

K_A = 1.5   #Schock factor


### Input: Geometry & Material
<img src="figures/GMA simplified.png" alt="GMA cross section" width="400"> <img src="figures/Lug Clevis.png" alt="GMA cross section" width="400"> 

In [177]:
# GEOMETRY
unit = {"length":"mm", "area":"mm²", "section mod":"mm³", "force":"N", "moment":"Nm", "stress": "MPa"}

geom = {
"bolt":     {"d": 0.6235*25.4,}, 
"lug":      {"d_lg": 0.625*25.4,   "t_lg": 0.567*25.4, "a_lg": 8.92, "c_lg": 8.92},
"clevis":   {"d_cl": 0.625*25.4,   "t_cl": 7.5,        "a_cl": 12, "c_cl": 17}}

mat = {"Rm_bt": 6.89476*(160 + 190) / 2, "Rm_lg": 1068, "Rm_cl": 1068, "Re_lg": 1000, "Re_cl": 1000}


### ANALYSIS: Bending Moment & Allowable bending failure stress

In [178]:
#Bending moment
if int_case == "c1":
    Mb_max = 1/8 * F_stat * (geom["lug"]["t_lg"]+ 2*geom["clevis"]["t_cl"])
elif int_case == "c2":
    Mb_max = 1/8 * F_stat * geom["lug"]["t_lg"]
elif int_case == "c3":
    Mb_max = 1/4 * F_stat * geom["clevis"]["t_cl"]
else:
    raise ValueError("Invalid integration case")

print(f"Mb_max = {Mb_max/1000:.1f} {unit['moment']}")

basis = mat["Rm_bt"] if bt_ht == "ht" else 400

#Bending stress

W = math.pi * geom["bolt"]["d"]**3 / 32
sigma_b = (K_A * Mb_max) / W

print(f"sigma_b = {sigma_b:.1f} {unit['moment']}")

#Allowable bending failure stress"
if ld_case == "stat":
    sigma_b_lim = 0.3 * basis
elif ld_case == "dyn_pul":
    sigma_b_lim = 0.2 * basis
elif ld_case == "dyn_al":
    sigma_b_lim = 0.15 * basis

print(f"sigma_b_lim = {sigma_b_lim} {unit["stress"]}")

Mb_max = 82.7 Nm
sigma_b = 318.1 Nm
sigma_b_lim = 361.97489999999993 MPa


### ANALYSIS: Shear

#### Shear bolt

In [179]:
A = math.pi *geom["bolt"]["d"]**2 / 4

# Shear bolt
tau_bt = (4/3) * (K_A * F_stat) / A

if ld_case == "stat":
    tau_bt_lim = 0.2 * mat["Rm_bt"]
elif ld_case == "dyn_pul":
    tau_bt_lim = 0.15 * mat["Rm_bt"]
elif ld_case == "dyn_al":
    tau_bt_lim = 0.1 * mat["Rm_bt"]
else:
    tau_bt_lim = 400

print(f"tau_bt = {tau_bt:.1f} {unit["stress"]}")
print(f"tau_bt_lim = {tau_bt_lim:.1f} {unit["stress"]}")

tau_bt = 228.4 MPa
tau_bt_lim = 241.3 MPa


#### Shear-out lug

In [180]:
tau_lg = (4/3) * (K_A * F_stat) / A

if ld_case == "stat":
    tau_lg_lim = 0.2 * mat["Rm_lg"]
elif ld_case == "dyn_pul":
    tau_lg_lim = 0.15 * mat["Rm_lg"]
elif ld_case == "dyn_al":
    tau_lg_lim = 0.1 * mat["Rm_lg"]
else:
    tau_lg_lim = 400

print(f"taub_lg = {tau_lg:.1f} {unit["stress"]}")
print(f"taub_lg_lim = {tau_lg_lim:.1f} {unit["stress"]}")

taub_lg = 228.4 MPa
taub_lg_lim = 213.6 MPa


#### Shear-out clevis

In [181]:
tau_cl = (4/3) * (K_A * F_stat / 2) / (A / 2)

if ld_case == "stat":
    tau_cl_lim = 0.2 * mat["Rm_cl"]
elif ld_case == "dyn_pul":
    tau_cl_lim = 0.15 * mat["Rm_cl"]
elif ld_case == "dyn_al":
    tau_cl_lim = 0.1 * mat["Rm_cl"]
else:
    tau_cl_lim = 400

print(f"taub_cl = {tau_cl:.1f} {unit["stress"]}")
print(f"taub_cl_lim = {tau_cl_lim:.1f} {unit["stress"]}")


taub_cl = 228.4 MPa
taub_cl_lim = 213.6 MPa


### ANALYSIS: Surface pressure

In [182]:
# Projected areas
A_lg_proj = 2 * geom["lug"]["d_lg"] * geom["lug"]["t_lg"]
A_cl_proj = 2 * geom["clevis"]["d_cl"] * geom["clevis"]["t_cl"]

print(A_lg_proj)
print(A_cl_proj)



457.2571499999999
238.125


#### Surface pressure bolt

In [183]:
# Bolt pressure
p_bt = K_A * F_stat / A_lg_proj

if ld_case == "stat":
    p_bt_lim = 0.35 * mat["Rm_bt"]
elif ld_case == "dyn_pul":
    p_bt_lim = 0.25 * mat["Rm_bt"]
elif ld_case == "dyn_al":
    p_bt_lim = 0.15 * mat["Rm_bt"]
else:
    p_bt_lim = 400


#### Surface pressure lug

In [184]:
# Lug pressure
p_lg = K_A * F_stat / A_lg_proj

if ld_case == "stat":
   p_lg_lim = 0.35 * mat["Rm_lg"]
elif ld_case == "dyn_pul":
    p_lg_lim = 0.25 * mat["Rm_lg"]
elif ld_case == "dyn_al":
    p_lg_lim = 0.15 * mat["Rm_lg"]
else:
    p_lg_lim = 400

#### Surface pressure clevis

In [185]:
p_cl = K_A * F_stat/2 / A_lg_proj

if ld_case == "stat":
    p_cl_lim = 0.35 * mat["Rm_cl"]
elif ld_case == "dyn_pul":
    p_cl_lim = 0.25 * mat["Rm_cl"]
elif ld_case == "dyn_al":
    p_cl_lim = 0.15 * mat["Rm_cl"]
else:
    p_cl_lim = 400


### ANALYSIS: Normal stress

In [186]:
sig_lg = ((K_A * F_stat) / (2 * geom["lug"]["c_lg"] * geom["lug"]["t_lg"])) * (1 + (3/2)*(geom["bolt"]["d"] / geom["lug"]["c_lg"] ))
sig_cl = ((K_A * F_stat * 0.5) / (2 * geom["clevis"]["c_cl"] * geom["clevis"]["t_cl"])) * (1 + (3/2)*(geom["bolt"]["d"] / geom["clevis"]["c_cl"]))

# Limits
if ld_case == "stat":
    sig_lg_lim = 0.5 * mat["Re_lg"] if mat == "ST" else 0.5 * mat["Rm_lg"]
    sig_cl_lim = 0.5 * mat["Re_cl"] if mat == "ST" else 0.5 * mat["Rm_cl"]
else:
    sig_lg_lim = 0.2 * mat["Re_lg"] if mat == "ST" else 0.2 * mat["Rm_lg"]
    sig_cl_lim = 0.2 * mat["Re_cl"] if mat == "ST" else 0.2 * mat["Rm_cl"]


### SUMMARY

In [187]:
print("\n=== RESULTS ===")
print(f"{'Name':<30} {'Parameter':<5} {'Value [MPa]':>15} {'Limit [MPa]':>15} {'Result':>10}")
print("-" * 90)

def check_line(name, parameter, value, limit):
    result = "OK" if value <= limit else "NOT OK"
    symbol = "✔" if result == "OK" else "✘"
    print(f"{name:<30} {parameter:<5} {value:>15.2f} {limit:>15.2f} {symbol:>10}")

check_line("Bending stress bolt", "sigma_b",sigma_b, sigma_b_lim)
check_line("Shear stress bolt","tau",tau_bt, tau_bt_lim)
check_line("Shear-Out stress lug","tau_lg",tau_lg, tau_lg_lim)
check_line("Shear-Out stress clevis","tau",tau_cl, tau_cl_lim)
check_line("Surface pressure bolt","p_cl",p_bt, p_bt_lim)
check_line("Surface pressure lug","p_lg",p_lg, p_lg_lim)
check_line("Surface pressure clevis","p_cl",p_cl, p_cl_lim)
check_line("Normal stress lug flank","sig_lg",sig_lg, sig_lg_lim)
check_line("Normal stress clevis flank","sig_cl",sig_cl, sig_cl_lim)


=== RESULTS ===
Name                           Parameter     Value [MPa]     Limit [MPa]     Result
------------------------------------------------------------------------------------------
Bending stress bolt            sigma_b          318.09          361.97          ✔
Shear stress bolt              tau            228.45          241.32          ✔
Shear-Out stress lug           tau_lg          228.45          213.60          ✘
Shear-Out stress clevis        tau            228.45          213.60          ✘
Surface pressure bolt          p_cl            73.81          422.30          ✔
Surface pressure lug           p_lg            73.81          373.80          ✔
Surface pressure clevis        p_cl            36.90          373.80          ✔
Normal stress lug flank        sig_lg          481.19          534.00          ✔
Normal stress clevis flank     sig_cl          158.65          534.00          ✔
